# LeetCode 1947 — Maximum Compatibility Score Sum bằng Python

Link đề: https://leetcode.com/problems/maximum-compatibility-score-sum/ · Độ khó: **Medium** · Chủ đề: Array, Dynamic Programming, Backtracking, Bit Manipulation, Bitmask

Skeleton LeetCode:

```python
class Solution:
    def maxCompatibilitySum(self, students: List[List[int]], mentors: List[List[int]]) -> int:
        ...
```

Lộ trình: **thử mọi hoán vị ghép cặp** `O(m! · m)` → **quay lui có cắt nhánh (cận trên)** → **DP bitmask trên tập mentor đã dùng** `O(m · 2^m)` (lời giải nộp) → **thuật toán Hungarian** `O(m³)` (bài toán phân công tổng quát).

## 1. Mục tiêu học

1. Tính **điểm tương thích** nhanh bằng bitmask: `n − popcount(a ⊕ b)` (số câu trả lời giống nhau).
2. Nhận ra bài toán **phân công (assignment problem)**: ghép `m` học sinh với `m` mentor (một–một) để tối đa tổng trọng số.
3. Cài **DP bitmask**: `dp[mask]` = tổng tốt nhất khi các mentor trong `mask` đã được ghép với `popcount(mask)` học sinh đầu tiên.
4. So sánh với vét cạn hoán vị và quay lui có cắt nhánh; biết thuật toán Hungarian cho `m` lớn.
5. Thấy vì sao tham lam "mỗi học sinh chọn mentor hợp nhất còn trống" sai, bằng phản ví dụ thật.

## 2. Đề bài & ví dụ

Một khảo sát có `n` câu hỏi, mỗi câu trả lời là `0` (không) hoặc `1` (có). Khảo sát được đưa cho `m` học sinh và `m` mentor (đánh số `0..m−1`). `students[i]` là câu trả lời của học sinh `i`, `mentors[j]` là của mentor `j`.

Mỗi học sinh được ghép với **một** mentor và mỗi mentor có **một** học sinh. **Điểm tương thích** của một cặp là số câu trả lời **giống nhau**. Ví dụ học sinh `[1,0,1]` và mentor `[0,0,1]` có điểm 2.

Tìm cách ghép để **tổng điểm tương thích lớn nhất**, trả về tổng đó.

**Ví dụ 1:** `students = [[1,1,0],[1,0,1],[0,0,1]]`, `mentors = [[1,0,0],[0,0,1],[1,1,0]]` → `8` (học sinh 0–mentor 2: 3, học sinh 1–mentor 0: 2, học sinh 2–mentor 1: 3).

**Ví dụ 2:** `students = [[0,0],[0,0],[0,0]]`, `mentors = [[1,1],[1,1],[1,1]]` → `0`.

**Ràng buộc:** `1 ≤ m, n ≤ 8`, mọi phần tử là 0 hoặc 1.

In [1]:
import random
import time
from functools import lru_cache
from itertools import permutations
from typing import List

examples = [
    ([[1, 1, 0], [1, 0, 1], [0, 0, 1]], [[1, 0, 0], [0, 0, 1], [1, 1, 0]], 8),
    ([[0, 0], [0, 0], [0, 0]], [[1, 1], [1, 1], [1, 1]], 0),
]

def compat(a, b):
    return sum(x == y for x, y in zip(a, b))

print("điểm [1,0,1] vs [0,0,1]:", compat([1, 0, 1], [0, 0, 1]))

điểm [1,0,1] vs [0,0,1]: 2


## 3. Kiến thức nền

### 3.1 Chuyển câu trả lời thành bitmask, điểm = `n − popcount(a ⊕ b)`

XOR bật bit ở câu trả lời **khác nhau**; số câu giống = `n − số bit 1 của XOR`. `int.bit_count()` (Python ≥ 3.10) hoặc `bin(x).count("1")`.

In [2]:
def to_mask(ans):
    m = 0
    for i, v in enumerate(ans):
        m |= v << i
    return m

a, b, n = to_mask([1, 0, 1]), to_mask([0, 0, 1]), 3
print(bin(a), bin(b), "giống:", n - (a ^ b).bit_count())

0b101 0b100 giống: 2


### 3.2 Ma trận điểm `score[i][j]`

Tính trước điểm của mọi cặp (học sinh `i`, mentor `j`) — `m² ≤ 64` cặp.

In [3]:
S, M, _ = examples[0]
score = [[compat(s, t) for t in M] for s in S]
for row in score:
    print(row)

[2, 0, 3]
[2, 2, 1]
[1, 3, 0]


### 3.3 Duyệt tập con bằng bitmask

`mask` với `m` bit biểu diễn tập mentor đã dùng. `mask >> j & 1` kiểm mentor `j`; `mask | 1 << j` thêm `j`; `mask.bit_count()` là số phần tử.

In [4]:
mask = 0b101
print([j for j in range(3) if mask >> j & 1], mask.bit_count(), bin(mask | 1 << 1))

[0, 2] 2 0b111


### 3.4 `itertools.permutations`

`permutations(range(m))` sinh mọi cách ghép: học sinh `i` ghép với mentor `perm[i]`.

In [5]:
print(list(permutations(range(3))))

[(0, 1, 2), (0, 2, 1), (1, 0, 2), (1, 2, 0), (2, 0, 1), (2, 1, 0)]


## 4. Ý tưởng nền tảng / trực giác

Ma trận điểm Ví dụ 1 (`score[i][j]`, hàng = học sinh, cột = mentor):

```
        m0  m1  m2
s0  [   2,  0,  3 ]
s1  [   2,  2,  1 ]   (ví dụ: s1=[1,0,1], m0=[1,0,0] giống 2 câu)
s2  [   1,  3,  0 ]
```

Cần chọn **một ô mỗi hàng, mỗi cột** để tổng lớn nhất. `s0 → m2 (3)`, `s1 → m0 (2)`, `s2 → m1 (3)` ⇒ 8.

Có `m! ≤ 40320` cách ghép — vét cạn được! Nhưng DP bitmask cho thấy cách "nén" tốt hơn: sau khi ghép `k` học sinh đầu, điều duy nhất quan trọng là **tập mentor đã dùng** (không quan trọng ai ghép với ai).

In [6]:
for row in [[compat(s, t) for t in examples[0][1]] for s in examples[0][0]]:
    print(row)

[2, 0, 3]
[2, 2, 1]
[1, 3, 0]


## 5. Cách 1 — Brute Force / baseline: mọi hoán vị

In [7]:
def max_compat_brute(students, mentors):
    m = len(students)
    score = [[compat(s, t) for t in mentors] for s in students]
    return max(sum(score[i][p[i]] for i in range(m)) for p in permutations(range(m)))

for S, M, want in examples:
    print(max_compat_brute(S, M), "| kỳ vọng", want)

8 | kỳ vọng 8
0 | kỳ vọng 0


**Độ phức tạp:** `O(m! · m)` = `40320 · 8 ≈ 3·10⁵` khi `m = 8` — **vẫn nhanh** với ràng buộc này. Nhưng nếu `m = 12` thì `~5·10⁹`. **Nút thắt:** hai hoán vị có cùng tập mentor cho `k` học sinh đầu tiên — phần còn lại tối ưu giống nhau — bị tính lại.

In [8]:
rng = random.Random(0)
def rand_case(m, n, rng):
    return ([[rng.randint(0, 1) for _ in range(n)] for _ in range(m)],
            [[rng.randint(0, 1) for _ in range(n)] for _ in range(m)])
S8, M8 = rand_case(8, 8, rng)
t = time.perf_counter(); r = max_compat_brute(S8, M8); dt = time.perf_counter() - t
print(f"m=8: {r} trong {1000*dt:.0f} ms; m=12 sẽ ≈ {dt * 12*11*10*9 * 1.5:.0f}s")

m=8: 45 trong 17 ms; m=12 sẽ ≈ 300s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight (DP bitmask):** ghép học sinh theo thứ tự `0, 1, …`. Trạng thái = tập `mask` mentor đã dùng; số học sinh đã ghép = `popcount(mask)`.

`dp[mask]` = tổng lớn nhất khi học sinh `0..k−1` (`k = popcount(mask)`) đã ghép với đúng các mentor trong `mask`.

**Chuyển:** từ `mask` (đã ghép `k` học sinh), ghép học sinh `k` với mentor `j ∉ mask`: `dp[mask | 1<<j] = max(…, dp[mask] + score[k][j])`.

**Đáp án:** `dp[(1 << m) − 1]`.

**Quy tắc "nếu… thì…":** *nếu* hai lịch sử ghép có cùng `mask` *thì* chỉ giữ lịch sử có tổng lớn hơn (phần tương lai không phụ thuộc ai ghép với ai).

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Quay lui có cắt nhánh (cận trên tham lam)

Ghép lần lượt từng học sinh; cắt nhánh khi `tổng hiện tại + cận trên phần còn lại ≤ tốt nhất`. Cận trên đơn giản: mỗi học sinh còn lại lấy điểm **lớn nhất** của mình (bỏ qua ràng buộc một–một).

In [9]:
def max_compat_backtrack(students, mentors):
    m = len(students)
    score = [[compat(s, t) for t in mentors] for s in students]
    best_row = [max(r) for r in score]
    suffix = [0] * (m + 1)
    for i in range(m - 1, -1, -1):
        suffix[i] = suffix[i + 1] + best_row[i]
    best = [0]; used = [False] * m; nodes = [0]
    def go(i, total):
        nodes[0] += 1
        if i == m:
            best[0] = max(best[0], total); return
        if total + suffix[i] <= best[0]:
            return                                        # không thể vượt kết quả tốt nhất
        for j in sorted(range(m), key=lambda j: -score[i][j]):    # thử mentor hợp nhất trước
            if not used[j]:
                used[j] = True; go(i + 1, total + score[i][j]); used[j] = False
    go(0, 0)
    return best[0], nodes[0]

for S, M, want in examples:
    print(max_compat_backtrack(S, M)[0], want)
print("số nút duyệt với m = 8:", max_compat_backtrack(S8, M8)[1], "(so với 8! = 40320 lá)")

8 8
0 0
số nút duyệt với m = 8: 41 (so với 8! = 40320 lá)


**Độ phức tạp:** worst-case vẫn `O(m!)`, nhưng cắt nhánh giảm mạnh trên dữ liệu thực tế.

### 7.2 Cách 3 — DP bitmask từ dưới lên (`O(m · 2^m)`) — lời giải chuẩn

In [10]:
def max_compat_dp_demo(students, mentors):
    m, n = len(students), len(students[0])
    sm = [to_mask(s) for s in students]; mm = [to_mask(t) for t in mentors]
    score = [[n - (a ^ b).bit_count() for b in mm] for a in sm]
    dp = [-1] * (1 << m); dp[0] = 0
    for mask in range(1 << m):
        if dp[mask] < 0: continue
        k = mask.bit_count()
        if k == m: continue
        for j in range(m):
            if not mask >> j & 1:
                nm = mask | 1 << j
                if dp[mask] + score[k][j] > dp[nm]:
                    dp[nm] = dp[mask] + score[k][j]
    for mask in range(1 << m):
        print(f"mask={mask:0{m}b} (mentor {[j for j in range(m) if mask >> j & 1]}): dp = {dp[mask]}")
    return dp[-1]

print("=>", max_compat_dp_demo(*examples[0][:2]))

mask=000 (mentor []): dp = 0
mask=001 (mentor [0]): dp = 2
mask=010 (mentor [1]): dp = 0
mask=011 (mentor [0, 1]): dp = 4
mask=100 (mentor [2]): dp = 3
mask=101 (mentor [0, 2]): dp = 5
mask=110 (mentor [1, 2]): dp = 5
mask=111 (mentor [0, 1, 2]): dp = 8
=> 8


In [11]:
class SolutionDP:
    def maxCompatibilitySum(self, students: List[List[int]], mentors: List[List[int]]) -> int:
        m, n = len(students), len(students[0])
        sm = [to_mask(s) for s in students]; mm = [to_mask(t) for t in mentors]
        score = [[n - (a ^ b).bit_count() for b in mm] for a in sm]
        full = (1 << m) - 1
        dp = [0] * (1 << m)
        for mask in range(1, full + 1):
            k = mask.bit_count() - 1                     # học sinh vừa được ghép: thứ k
            best = 0
            sub = mask
            while sub:
                j = (sub & -sub).bit_length() - 1        # một mentor trong mask
                cand = dp[mask ^ (1 << j)] + score[k][j]
                if cand > best: best = cand
                sub &= sub - 1
            dp[mask] = best
        return dp[full]

for S, M, want in examples:
    print(SolutionDP().maxCompatibilitySum(S, M), want)

8 8
0 0


Bản nộp viết theo kiểu "kéo" (pull): `dp[mask] = max_{j ∈ mask} dp[mask ∖ j] + score[k][j]` với `k = popcount(mask) − 1` — học sinh **cuối cùng** trong nhóm ghép với mentor `j`. **Độ phức tạp:** `2^m` trạng thái × `≤ m` chuyển ⇒ `O(m · 2^m) = 8 · 256` — rất nhỏ; bộ nhớ `O(2^m)`.

### 7.3 Cách 4 — Thuật toán Hungarian (Kuhn–Munkres) `O(m³)`

Bài toán phân công tổng quát (ma trận chi phí `m × m`) có thuật toán đa thức. Ta tối đa hóa ⇒ tối thiểu hóa chi phí `−score`. Cài đặt chuẩn với thế vị (potentials) `u`, `v`.

In [12]:
def hungarian_min(cost):
    n = len(cost)
    INF = float("inf")
    u = [0] * (n + 1); v = [0] * (n + 1); p = [0] * (n + 1); way = [0] * (n + 1)
    for i in range(1, n + 1):
        p[0] = i; j0 = 0
        minv = [INF] * (n + 1); used = [False] * (n + 1)
        while True:
            used[j0] = True; i0 = p[j0]; delta = INF; j1 = 0
            for j in range(1, n + 1):
                if not used[j]:
                    cur = cost[i0 - 1][j - 1] - u[i0] - v[j]
                    if cur < minv[j]: minv[j] = cur; way[j] = j0
                    if minv[j] < delta: delta = minv[j]; j1 = j
            for j in range(n + 1):
                if used[j]: u[p[j]] += delta; v[j] -= delta
                else: minv[j] -= delta
            j0 = j1
            if p[j0] == 0: break
        while True:
            j1 = way[j0]; p[j0] = p[j1]; j0 = j1
            if j0 == 0: break
    assign = [0] * n                                     # assign[i] = cột ghép với hàng i
    for j in range(1, n + 1):
        assign[p[j] - 1] = j - 1
    return sum(cost[i][assign[i]] for i in range(n)), assign

class SolutionHungarian:
    def maxCompatibilitySum(self, students: List[List[int]], mentors: List[List[int]]) -> int:
        score = [[compat(s, t) for t in mentors] for s in students]
        total, _ = hungarian_min([[-x for x in row] for row in score])
        return -total

for S, M, want in examples:
    print(SolutionHungarian().maxCompatibilitySum(S, M), want)

8 8
0 0


**Độ phức tạp:** `O(m³)` — với `m = 8` không nhanh hơn DP, nhưng là lựa chọn duy nhất khi `m` lên hàng trăm (DP `2^m` bất khả). Ghi nhớ: DP bitmask cho `m ≤ ~20`, Hungarian cho `m` lớn.

## 8. TRACE từng bước

Trace DP (kiểu "kéo") trên Ví dụ 1 với `score = [[2,0,3],[2,2,1],[1,3,0]]` (chỉ các `mask` quan trọng):

| `mask` | `k` (học sinh cuối) | ứng viên `dp[mask∖j] + score[k][j]` | `dp[mask]` |
|---|---|---|---|
| `001` {m0} | 0 | `0 + 2` | 2 |
| `010` {m1} | 0 | `0 + 0` | 0 |
| `100` {m2} | 0 | `0 + 3` | 3 |
| `101` {m0,m2} | 1 | `dp[100] + s[1][0] = 3+2`, `dp[001] + s[1][2] = 2+1` | 5 |
| `111` | 2 | `dp[110] + s[2][0]`, `dp[101] + s[2][1] = 5 + 3 = 8`, `dp[011] + s[2][2]` | **8** |

In [13]:
def trace_dp_pull(students, mentors):
    m, n = len(students), len(students[0])
    score = [[compat(s, t) for t in mentors] for s in students]
    dp = [0] * (1 << m)
    for mask in range(1, 1 << m):
        k = mask.bit_count() - 1
        cands = [(dp[mask ^ (1 << j)] + score[k][j], j) for j in range(m) if mask >> j & 1]
        dp[mask] = max(c for c, _ in cands)
        print(f"mask={mask:0{m}b} k={k}: " + ", ".join(f"m{j}->{c}" for c, j in cands) + f" => {dp[mask]}")
    return dp[-1]

trace_dp_pull(*examples[0][:2])

mask=001 k=0: m0->2 => 2
mask=010 k=0: m1->0 => 0
mask=011 k=1: m0->2, m1->4 => 4
mask=100 k=0: m2->3 => 3
mask=101 k=1: m0->5, m2->3 => 5
mask=110 k=1: m1->5, m2->1 => 5
mask=111 k=2: m0->6, m1->8, m2->4 => 8


8

Truy vết **cách ghép** tối ưu từ bảng DP:

In [14]:
def best_assignment(students, mentors):
    m = len(students)
    score = [[compat(s, t) for t in mentors] for s in students]
    dp = [0] * (1 << m)
    for mask in range(1, 1 << m):
        k = mask.bit_count() - 1
        dp[mask] = max(dp[mask ^ (1 << j)] + score[k][j] for j in range(m) if mask >> j & 1)
    mask, pair = (1 << m) - 1, [0] * m
    while mask:
        k = mask.bit_count() - 1
        j = next(j for j in range(m) if mask >> j & 1 and dp[mask ^ (1 << j)] + score[k][j] == dp[mask])
        pair[k] = j; mask ^= 1 << j
    return dp[-1], pair

v, pair = best_assignment(*examples[0][:2])
print("tổng", v, "| học sinh i -> mentor", pair)

tổng 8 | học sinh i -> mentor [2, 0, 1]


## 9. Code nộp LeetCode

In [15]:
from typing import List

class Solution:
    def maxCompatibilitySum(self, students: List[List[int]], mentors: List[List[int]]) -> int:
        m, n = len(students), len(students[0])
        to_bits = lambda arr: sum(v << i for i, v in enumerate(arr))
        sm = [to_bits(s) for s in students]
        mm = [to_bits(t) for t in mentors]
        score = [[n - bin(a ^ b).count("1") for b in mm] for a in sm]   # số câu giống nhau
        dp = [0] * (1 << m)                     # dp[mask]: các mentor trong mask đã ghép với popcount(mask) học sinh đầu
        for mask in range(1, 1 << m):
            k = bin(mask).count("1") - 1        # học sinh thứ k ghép cuối cùng
            dp[mask] = max(dp[mask ^ (1 << j)] + score[k][j] for j in range(m) if mask >> j & 1)
        return dp[-1]

for S, M, want in examples:
    assert Solution().maxCompatibilitySum(S, M) == want
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

**Định nghĩa:** `D(mask)` = tổng lớn nhất trên mọi cách ghép **song ánh** giữa học sinh `0..k−1` (`k = |mask|`) và tập mentor `mask`.

**Truy hồi:** trong một cách ghép tối ưu như vậy, học sinh `k−1` ghép với một mentor `j ∈ mask`; phần còn lại là cách ghép học sinh `0..k−2` với `mask ∖ {j}`, và phải tối ưu (nếu không, thay bằng cách tốt hơn sẽ tăng tổng). Vậy `D(mask) = max_{j ∈ mask} D(mask ∖ {j}) + score[k−1][j]`, với `D(∅) = 0`. Duyệt `mask` tăng dần bảo đảm `mask ∖ {j} < mask` đã tính.

**Đáp án:** `D(toàn bộ)` — mọi cách ghép đầy đủ đều được xét (mỗi hoán vị ứng với một chuỗi `mask` lồng nhau).

**Điểm tương thích:** câu `t` giống nhau ⇔ bit `t` của `a ⊕ b` bằng 0 ⇒ số câu giống = `n − popcount(a ⊕ b)`.

## 11. Phân tích Time / Space

| Cách | Time | Space |
|---|---|---|
| 1. Hoán vị | `O(m! · m)` | `O(m²)` |
| 2. Quay lui + cắt nhánh | `O(m!)` worst, thực tế ít hơn nhiều | `O(m)` |
| 3. DP bitmask | `O(m · 2^m + m² · n)` | `O(2^m)` |
| 4. Hungarian | `O(m³)` | `O(m²)` |

In [16]:
rng = random.Random(1947)
for m in [8, 12, 16]:
    S, M = rand_case(m, 8, rng)
    res = []
    for name, f in [("DP/Nộp", Solution().maxCompatibilitySum), ("Hungarian", SolutionHungarian().maxCompatibilitySum)]:
        t = time.perf_counter(); r = f(S, M); res.append(r)
        print(f"m={m} {name}: {r} trong {1000*(time.perf_counter()-t):.1f} ms")
    if m <= 8:
        t = time.perf_counter(); res.append(max_compat_brute(S, M)); print(f"m={m} hoán vị: {1000*(time.perf_counter()-t):.0f} ms")
    assert len(set(res)) == 1

m=8 DP/Nộp: 44 trong 0.2 ms
m=8 Hungarian: 44 trong 0.1 ms
m=8 hoán vị: 17 ms
m=12 DP/Nộp: 76 trong 4.0 ms
m=12 Hungarian: 76 trong 0.1 ms
m=16 DP/Nộp: 96 trong 78.5 ms
m=16 Hungarian: 96 trong 0.2 ms


(`m = 12, 16` vượt ràng buộc đề — minh họa DP `2^m` vẫn ổn tới khoảng 20, còn hoán vị thì không.)

## 12. Bộ test

### 12.1 Ví dụ đề + edge case (tự dựng, đối chiếu hoán vị)

In [17]:
sols = [Solution(), SolutionDP(), SolutionHungarian()]
edge = examples + [
    ([[1]], [[1]], 1),
    ([[1]], [[0]], 0),
    ([[1, 0], [0, 1]], [[0, 1], [1, 0]], 4),          # ghép chéo
    ([[1, 1, 1], [0, 0, 0]], [[0, 0, 0], [1, 1, 1]], 6),
    ([[1] * 8] * 8, [[1] * 8] * 8, 64),
]
for S, M, want in edge:
    assert max_compat_brute(S, M) == want
    for s in sols:
        assert s.maxCompatibilitySum(S, M) == want, (S, M, s)
    assert max_compat_backtrack(S, M)[0] == want
    print("OK", len(S), "học sinh ->", want)

OK 3 học sinh -> 8
OK 3 học sinh -> 0
OK 1 học sinh -> 1
OK 1 học sinh -> 0
OK 2 học sinh -> 4
OK 2 học sinh -> 6
OK 8 học sinh -> 64


### 12.2 Ngẫu nhiên có seed: mọi kích thước trong ràng buộc

In [18]:
rng = random.Random(42)
for _ in range(300):
    m, n = rng.randint(1, 7), rng.randint(1, 8)
    S, M = rand_case(m, n, rng)
    want = max_compat_brute(S, M)
    for s in sols:
        assert s.maxCompatibilitySum(S, M) == want
    assert max_compat_backtrack(S, M)[0] == want
    v, pair = best_assignment(S, M)
    assert sorted(pair) == list(range(m)) and sum(compat(S[i], M[pair[i]]) for i in range(m)) == want
print("300 bộ ngẫu nhiên: 5 cách khớp hoán vị; cách ghép truy vết hợp lệ")

300 bộ ngẫu nhiên: 5 cách khớp hoán vị; cách ghép truy vết hợp lệ


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Tham lam: mỗi học sinh chọn mentor hợp nhất còn trống

In [19]:
def wrong_greedy(students, mentors):
    m = len(students); used = set(); total = 0
    for s in students:
        j = max((j for j in range(m) if j not in used), key=lambda j: compat(s, mentors[j]))
        used.add(j); total += compat(s, mentors[j])
    return total

rng = random.Random(5)
for _ in range(2000):
    S, M = rand_case(3, 3, rng)
    if wrong_greedy(S, M) != Solution().maxCompatibilitySum(S, M):
        print("phản ví dụ:", S, M, "| tham lam:", wrong_greedy(S, M), "| đúng:", Solution().maxCompatibilitySum(S, M))
        break

phản ví dụ: [[0, 0, 1], [1, 0, 1], [1, 0, 0]] [[1, 0, 1], [1, 0, 1], [0, 1, 1]] | tham lam: 5 | đúng: 7


**Vì sao sai:** học sinh đầu "giành" mentor tốt nhất của mình, có thể chính là mentor **duy nhất** hợp với học sinh sau ⇒ tổng nhỏ hơn. Đây là bài toán phân công — cần DP/Hungarian.

### Lỗi 2 — Đếm câu **khác nhau** thay vì giống nhau

In [20]:
def wrong_diff_score(students, mentors):
    m = len(students)
    score = [[sum(x != y for x, y in zip(s, t)) for t in mentors] for s in students]   # SAI
    return max(sum(score[i][p[i]] for i in range(m)) for p in permutations(range(m)))

for S, M, want in examples:
    print("sai:", wrong_diff_score(S, M), "| đúng:", want)

sai: 7 | đúng: 8
sai: 6 | đúng: 0


**Vì sao sai:** `popcount(a ⊕ b)` là số câu **khác**; điểm tương thích là `n − popcount(a ⊕ b)`. (Tối đa hóa "khác nhau" là bài toán ngược.)

### Lỗi 3 — DP lấy chỉ số học sinh sai (không gắn với `popcount(mask)`)

In [21]:
def wrong_dp_index(students, mentors):
    m = len(students)
    score = [[compat(s, t) for t in mentors] for s in students]
    dp = [0] * (1 << m)
    for mask in range(1, 1 << m):
        k = (mask.bit_length() - 1)                       # SAI: bit cao nhất, không phải số học sinh đã ghép
        dp[mask] = max(dp[mask ^ (1 << j)] + score[min(k, m - 1)][j] for j in range(m) if mask >> j & 1)
    return dp[-1]

for S, M, want in examples + [([[1, 0], [0, 1]], [[0, 1], [1, 0]], 4)]:
    print("sai:", wrong_dp_index(S, M), "| đúng:", Solution().maxCompatibilitySum(S, M))

sai: 5 | đúng: 8
sai: 0 | đúng: 0
sai: 2 | đúng: 4


**Vì sao sai:** trạng thái `mask` nghĩa là "**`popcount(mask)`** học sinh đầu tiên đã ghép với các mentor trong `mask`". Dùng bit cao nhất (hay chỉ số mentor) làm chỉ số học sinh khiến một học sinh có thể bị ghép nhiều lần còn người khác không được ghép. **Sửa:** `k = popcount(mask) − 1`.

### Lỗi 4 — Duyệt `mask` sai thứ tự trong DP "đẩy" (push) tại chỗ

In [22]:
def wrong_push_order(students, mentors):
    m = len(students)
    score = [[compat(s, t) for t in mentors] for s in students]
    dp = [-1] * (1 << m); dp[0] = 0
    for mask in range((1 << m) - 1, -1, -1):              # SAI: duyệt giảm dần, mask con chưa được tính
        if dp[mask] < 0: continue
        k = mask.bit_count()
        for j in range(m):
            if k < m and not mask >> j & 1:
                dp[mask | 1 << j] = max(dp[mask | 1 << j], dp[mask] + score[k][j])
    return dp[-1]

for S, M, want in examples:
    print("sai:", wrong_push_order(S, M), "| đúng:", want)

sai: -1 | đúng: 8
sai: -1 | đúng: 0


**Vì sao sai:** kiểu "đẩy" cần `dp[mask]` **hoàn chỉnh** trước khi đẩy sang `mask ∪ {j}` (lớn hơn). Duyệt giảm dần thì `dp[mask]` lớn được xét trước khi các `mask` con đẩy tới ⇒ hầu hết vẫn `−1`. **Sửa:** duyệt `mask` **tăng dần**.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Ghép một–một hai nhóm `m ≤ ~20` phần tử, tối ưu tổng"** ⇒ **DP bitmask** `dp[mask]`, phần tử nhóm 1 lấy theo `popcount(mask)`.
- `m` lớn (hàng trăm) ⇒ **Hungarian** `O(m³)` / min-cost flow.
- Vector nhị phân ⇒ bitmask + XOR + popcount để so sánh nhanh.
- `m ≤ 8` ⇒ hoán vị `m!` cũng qua — nhưng DP là kỹ năng tổng quát hơn.
- Bài tương tự: LeetCode 1066 *Campus Bikes II*, 1879 *Minimum XOR Sum of Two Arrays*, 2172 *Maximum AND Sum of Array*.

## 15. Biến thể của bài

1. **LeetCode 1879 — Minimum XOR Sum of Two Arrays:** cùng DP bitmask, tối thiểu `Σ (a_i XOR b_{p(i)})`.
2. **Số học sinh ≠ số mentor** (`m₁ ≤ m₂`) ⇒ `mask` trên nhóm lớn, chỉ xét `popcount(mask) ≤ m₁`.
3. **Mỗi mentor nhận tối đa 2 học sinh** (LeetCode 2172) ⇒ mask cơ số 3.
4. **Đếm số cách ghép đạt tối ưu** ⇒ DP song song đếm số cách.
5. **`m` lớn** ⇒ Hungarian (đã cài ở Cách 4) hoặc luồng chi phí nhỏ nhất.

## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hoán vị | thử mọi cách ghép | `O(m!·m)` | `O(m²)` | `m ≤ 9`, kiểm chứng |
| 2. Quay lui + cắt nhánh | DFS với cận trên | `O(m!)` worst | `O(m)` | khi cận tốt, dữ liệu "dễ" |
| 3. DP bitmask | `dp[mask]` | `O(m·2^m)` | `O(2^m)` | **nộp bài**, `m ≤ 20` |
| 4. Hungarian | thế vị, đường tăng | `O(m³)` | `O(m²)` | `m` lớn |

## 17. 5 bài tập từ dễ đến khó

**Bài 1 (đoán output):** `students = [[1,0],[0,1]]`, `mentors = [[1,0],[0,1]]`; và `mentors` đảo thứ tự?

**Bài 2 (tính tay):** Lập ma trận `score` cho Ví dụ 1 và kiểm `dp[101] = 5`.

**Bài 3 (điền TODO):** Viết `count_optimal(students, mentors)` — số cách ghép đạt tổng lớn nhất (DP đếm song song).

**Bài 4 (tự viết lại):** Viết `min_xor_sum(a, b)` (LeetCode 1879): ghép `a[i]` với `b[p(i)]` để tối thiểu `Σ a[i] XOR b[p(i)]`.

**Bài 5 (mini challenge):** Số học sinh `m₁` ít hơn số mentor `m₂` (`m₁ ≤ m₂ ≤ 10`): mỗi học sinh một mentor khác nhau, mentor có thể không có ai. Viết `max_compat_unequal(students, mentors)`.

In [23]:
def count_optimal(students, mentors):
    # TODO
    pass

def min_xor_sum(a, b):
    # TODO
    pass

def max_compat_unequal(students, mentors):
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

| Bài | Gợi ý 1 | Gợi ý 2 | Gợi ý 3 |
|---|---|---|---|
| 1 | Ghép thẳng | Mỗi cặp giống 2 câu ⇒ 4 | Đảo thứ tự mentor vẫn 4 (ghép chéo) |
| 2 | `score = [[2,0,3],[2,2,1],[1,3,0]]` | `dp[100] = 3`, `dp[001] = 2` | `max(3 + 2, 2 + 1) = 5` |
| 3 | Giữ `cnt[mask]` | Khi cập nhật bằng giá trị lớn hơn: gán; bằng: cộng | `cnt[0] = 1` |
| 4 | Như bài gốc nhưng `min` và chi phí `a ^ b` | `dp[mask] = min_j …` | Khởi tạo `dp[0] = 0` |
| 5 | `mask` trên mentor (`m₂` bit) | Chỉ xét `popcount(mask) ≤ m₁` | Đáp án = max `dp[mask]` với `popcount = m₁` |

## 19. Đáp án tham khảo

In [24]:
r1 = [Solution().maxCompatibilitySum([[1, 0], [0, 1]], [[1, 0], [0, 1]]),
      Solution().maxCompatibilitySum([[1, 0], [0, 1]], [[0, 1], [1, 0]])]
print("Bài 1:", r1); assert r1 == [4, 4]
print("Bài 2:"); assert trace_dp_pull(*examples[0][:2]) == 8

Bài 1: [4, 4]
Bài 2:
mask=001 k=0: m0->2 => 2
mask=010 k=0: m1->0 => 0
mask=011 k=1: m0->2, m1->4 => 4
mask=100 k=0: m2->3 => 3
mask=101 k=1: m0->5, m2->3 => 5
mask=110 k=1: m1->5, m2->1 => 5
mask=111 k=2: m0->6, m1->8, m2->4 => 8


In [25]:
def count_optimal(students, mentors):
    m = len(students); score = [[compat(s, t) for t in mentors] for s in students]
    dp = [0] * (1 << m); cnt = [0] * (1 << m); cnt[0] = 1
    for mask in range(1, 1 << m):
        k = mask.bit_count() - 1; best = -1; c = 0
        for j in range(m):
            if mask >> j & 1:
                v = dp[mask ^ (1 << j)] + score[k][j]
                if v > best: best, c = v, cnt[mask ^ (1 << j)]
                elif v == best: c += cnt[mask ^ (1 << j)]
        dp[mask], cnt[mask] = best, c
    return cnt[-1]

def min_xor_sum(a, b):
    m = len(a); dp = [0] * (1 << m)
    for mask in range(1, 1 << m):
        k = mask.bit_count() - 1
        dp[mask] = min(dp[mask ^ (1 << j)] + (a[k] ^ b[j]) for j in range(m) if mask >> j & 1)
    return dp[-1]

rng = random.Random(3)
for _ in range(200):
    m = rng.randint(1, 6); S, M = rand_case(m, rng.randint(1, 5), rng)
    sc = [sum(compat(S[i], M[p[i]]) for i in range(m)) for p in permutations(range(m))]
    assert count_optimal(S, M) == sc.count(max(sc))
    a = [rng.randint(0, 20) for _ in range(m)]; b = [rng.randint(0, 20) for _ in range(m)]
    assert min_xor_sum(a, b) == min(sum(a[i] ^ b[p[i]] for i in range(m)) for p in permutations(range(m)))
assert min_xor_sum([1, 2], [2, 3]) == 2 and min_xor_sum([1, 0, 3], [5, 3, 4]) == 8      # ví dụ của LeetCode 1879
print("Bài 3, 4: OK", count_optimal(*examples[0][:2]), count_optimal(*examples[1][:2]))

Bài 3, 4: OK 1 6


In [26]:
def max_compat_unequal(students, mentors):
    m1, m2 = len(students), len(mentors)
    score = [[compat(s, t) for t in mentors] for s in students]
    NEG = -1
    dp = [NEG] * (1 << m2); dp[0] = 0; best = 0 if m1 == 0 else NEG
    for mask in range(1 << m2):
        if dp[mask] < 0: continue
        k = mask.bit_count()
        if k == m1:
            best = max(best, dp[mask]); continue
        for j in range(m2):
            if not mask >> j & 1:
                nm = mask | 1 << j
                dp[nm] = max(dp[nm], dp[mask] + score[k][j])
    return best

rng = random.Random(9)
for _ in range(200):
    m1 = rng.randint(1, 4); m2 = rng.randint(m1, 6); n = rng.randint(1, 5)
    S = [[rng.randint(0, 1) for _ in range(n)] for _ in range(m1)]
    M = [[rng.randint(0, 1) for _ in range(n)] for _ in range(m2)]
    brute = max(sum(compat(S[i], M[p[i]]) for i in range(m1)) for p in permutations(range(m2), m1))
    assert max_compat_unequal(S, M) == brute
print("Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK")

Bài 5 (biến thể tự dựng — đối chiếu lại đề gốc nếu có): OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
score = [[n - bin(a ^ b).count("1") for b in mentor_bits] for a in student_bits]
dp = [0] * (1 << m)
for mask in range(1, 1 << m):                       # tăng dần
    k = bin(mask).count("1") - 1                    # học sinh thứ k ghép cuối
    dp[mask] = max(dp[mask ^ (1 << j)] + score[k][j] for j in range(m) if mask >> j & 1)
return dp[-1]
```

**Checklist:**
- [ ] Điểm = số câu **giống** (`n − popcount(XOR)`)?
- [ ] Không dùng tham lam ghép từng học sinh?
- [ ] Chỉ số học sinh = `popcount(mask) − 1` (kiểu kéo) hoặc `popcount(mask)` (kiểu đẩy)?
- [ ] Duyệt `mask` tăng dần?
- [ ] Trả `dp[(1 << m) − 1]`.